<div align="right" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img
 src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/alx-courses/aice/assets/Content_page_banner_blue_dots.png"
 alt="ALX Content Header"
 class="full-width-image"
/>
</div>

# Ensuring data integrity with transactions

In this train, we'll delve into the concept of transactions in SQL, a critical component for maintaining data integrity and consistency across database operations. We'll explore how to use transactions to wrap stored procedure calls, ensuring that changes to the database are made reliably and safely.

> ⚠️ Ensure that you have downloaded the database file `chinook.db` and saved it in the same directory as this notebook.

## Learning objectives

By the end of this train, you should be able to:
- Define what transactions are and explain their importance in database operations, particularly in relation to stored procedures.
- Implement transactions to manage a series of SQL operations, ensuring either complete success or rollback in case of failure.
- Understand the impact of transactions on database performance and learn best practices for their use to maintain efficiency.
- Use transactions to ensure data consistency and integrity when executing multiple related operations, such as updates, inserts, or deletes.

## Overview

Transactions are a fundamental concept in database management, ensuring that **a group of SQL operations** within a single process are **completed successfully**. If any operation within the transaction fails, the entire transaction is rolled back, meaning no changes are made to the database. This mechanism is crucial for maintaining the consistency and integrity of data, especially in complex operations involving multiple steps.

## Exploring the database schema

Python's standard library for SQLite, `ipython_sql`,  does not support full transactional operations. Therefore, we'll use `sqlite3` instead. If the import below fails, uncomment the first line to install it with `pip`. 

In [2]:
# !pip install sqlite3
import sqlite3
# Connect to the Chinook database
conn = sqlite3.connect('chinook.db')

Understanding the database schema is crucial before diving into transactions. Here is a view of all of our tables in the database:

<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://github.com/Explore-AI/Pictures/blob/master/sqlite-sample-database-color.jpg?raw=true"  style="width:50%";/>
<br>
<br>
    <em>Figure 1: Chinook ERD</em>
</div>


In [3]:
# List the tables available in Chinook database
cursor = conn.cursor()

cursor.execute("""
SELECT 
    name 
FROM 
    sqlite_master 
WHERE 
    type='table';
""")

tables = cursor.fetchall()
print("Tables in the Chinook database:", tables)

Tables in the Chinook database: [('albums',), ('sqlite_sequence',), ('artists',), ('customers',), ('employees',), ('genres',), ('invoices',), ('invoice_items',), ('media_types',), ('playlists',), ('playlist_track',), ('tracks',), ('sqlite_stat1',)]


## Transactions in SQL
Transactions in SQL are used to **group a set of operations so that they are executed as a single unit**. This all-or-nothing approach ensures that if any part of the transaction **fails**, the entire transaction is **rolled back**, leaving the database in its original state before the transaction began.

### Why transactions?
Imagine a scenario in the Chinook database where we're updating the stock levels of various tracks after a sale. Each update operation decreases the stock by the quantity sold. Without transactions, if our application crashes or encounters an error after updating some but not all of the tracks, our database would be left in an inconsistent state – some stock levels would be updated, and others wouldn't. Transactions prevent this by ensuring that either all updates succeed or none at all, maintaining data integrity.

### Starting a transaction

In most SQL databases, a transaction is started with the `BEGIN TRANSACTION` statement. Operations that form part of the transaction follow, and if everything goes as planned, the transaction is committed using `COMMIT`. If something goes wrong, the transaction can be rolled back using `ROLLBACK`.

In [4]:
cursor.execute("BEGIN TRANSACTION;")
# Perform operations here
    # SELECT...

# If all operations succeed.
cursor.execute("COMMIT;")

# or
conn.commit()

# If an operation fails.
cursor.execute("ROLLBACK;")

# or
conn.rollback()

OperationalError: cannot rollback - no transaction is active

## Example 1: Basic transaction for inserting new records

Let's start with a transaction that inserts a new artist into the `artists` table and their album into the `albums` table in the Chinook database. We will need two SQL statements to do this. Using a transaction ensures both operations are executed as a single unit. 

Let's suppose we made a mistake and thought the `albums` table was called `albooms`. When we run the transaction, we will insert a new artist, but when we try to insert the album into `albooms` that does not exist, it creates an error because one of the statements is incorrect.

In [5]:
# Start the transaction
cursor.execute("BEGIN TRANSACTION;")

# Insert a new artist
cursor.execute("INSERT INTO artists (Name) VALUES ('New Artist');")

# Get the id of the last entry in artists
artist_id = cursor.lastrowid

# Insert a new album for the new artist in the incorrectly named albooms table.
cursor.execute("INSERT INTO albooms (Title, ArtistId) VALUES (?, ?);", ('New Album', artist_id))


OperationalError: no such table: albooms

The code above:
* Begins a transaction explicitly with `BEGIN TRANSACTION;`. 
* Inserts a new artist named "New Artist" into the `artists` table and a new album titled "New Album" associated with the newly inserted artist into the incorrectly named `albooms`` table.

Even though we got an error, the transaction is open and we could add an artist:

In [6]:
cursor.execute("SELECT * FROM artists WHERE Name = 'New Artist'")
cursor.fetchall()

[(276, 'New Artist')]

But, if we check if there is a new album inserted, we get no results set:

In [7]:
cursor.execute("SELECT * FROM albums WHERE Title = 'New Album'")
cursor.fetchall()

[]

We now have two choices: `COMMIT` these incorrect changes to the database or `ROLLBACK` to the database state before we started the transaction. If we commit, we will have a new artist but no information about the album. 

In [8]:
# Do not run this cell
cursor.execute("COMMIT;")

Or we can `ROLLBACK`, fix the mistake, and run the query again. 

In [9]:
cursor.execute("ROLLBACK;")

OperationalError: cannot rollback - no transaction is active

In [10]:
# Start the transaction
cursor.execute("BEGIN TRANSACTION;")

# Insert a new artist
cursor.execute("INSERT INTO artists (Name) VALUES ('New Artist');")
artist_id = cursor.lastrowid

# Insert a new album for the new artist in the correct table
cursor.execute("INSERT INTO albums (Title, ArtistId) VALUES (?, ?);", ('New Album', artist_id))

Run the `SELECT` statements again to check if both changes have been made, and once we're sure both `INSERT`s worked correctly, we can `COMMIT` the changes, which closes the transaction.

> ⚠️ Note, when we close the transaction with a `COMMIT` statement, the changes will be applied, and we cannot `ROLLBACK` the changes. 

In [ ]:
cursor.execute("COMMIT;")

## Example 2: Updating records in a transaction

Next, let's suppose this business would like to have a sale, where all tracks in the 'Pop' genre are discounted by `10%`. The goal is to use a transaction to apply multiple updates consistently.

Let's start by displaying the first two records of the Pop genre so that we can later confirm that our transaction was executed successfully.

In [11]:
import pandas as pd
# Output the original UnitPrice for tracks in the 'Pop' genre
cursor.execute("""
SELECT 
    TrackId,
    Name, 
    UnitPrice 
FROM 
    tracks 
WHERE 
    GenreId = (SELECT GenreId FROM genres WHERE Name = 'Pop')
LIMIT 2;
""")

# Fetch all the rows
cursor.fetchall()

ModuleNotFoundError: No module named 'pandas'

We'll then go ahead and make the discount changes.

In [ ]:
# Start the transaction
conn.execute("BEGIN TRANSACTION;")

# Update the UnitPrice for tracks in the 'Pop' genre by 10%
cursor.execute("""
UPDATE 
    tracks 
SET 
    UnitPrice = UnitPrice * (1 - 0.1)
WHERE 
    GenreId = (SELECT GenreId FROM genres WHERE Name = 'Pop');
""")


Let's finish by confirming that our changes were successful.

In [ ]:
# Output the results of the updated UnitPrice for tracks in the 'Pop' genre
cursor.execute("""
SELECT 
    TrackId,
    Name, 
    UnitPrice 
FROM 
    tracks 
WHERE 
    GenreId = (SELECT GenreId FROM genres WHERE Name = 'Pop')
LIMIT 2;
""")

# Fetch all the rows
updated_prices = cursor.fetchall()

# Print each updated track price
for track in updated_prices:
    print(f"Track ID: {track[0]}, Name: {track[1]}, UnitPrice: {track[2]}")

Once we're sure all the changes have been made, we can use `COMMIT`.

In [ ]:
cursor.execute("COMMIT;")

## Example 3: Assigning tracks to a new playlist

Suppose we've been tasked with creating a new playlist for a promotional event, adding a selection of tracks to it. This involves creating the playlist, identifying tracks of the rock genre, and then associating these tracks with the new playlist.

Steps involved:
* Create a new playlist named "Promotional Hits".
* Select tracks from the "Rock" genre.
* Insert entries into the `playlist_track` table to link the selected tracks with the new playlist.


In [ ]:
# Step 1: Create a new playlist
cursor.execute("INSERT INTO playlists (Name) VALUES ('Promotional Hits');")
playlist_id = cursor.lastrowid  # Get the new playlist ID

# Step 2: Identify tracks from the "Rock" genre
cursor.execute("""
SELECT 
    TrackId 
FROM 
    tracks 
WHERE 
    GenreId = (SELECT GenreId FROM genres WHERE Name = 'Rock');
""")
track_ids = cursor.fetchall()  # Fetch all matching track IDs

# Step 3: Associate tracks with the new playlist
for track_id in track_ids:
    cursor.execute("INSERT INTO playlist_track (PlaylistId, TrackId) VALUES (?, ?);", (playlist_id, track_id[0])) # Here we can use (?,?) to use (playlist_id, track_id[0]) in the query. This is not something we need to remember.



# Step 3: Verification: List tracks in the "Promotional Hits" playlist
cursor.execute("""
SELECT 
    tracks.Name 
FROM 
    tracks
JOIN 
    playlist_track 
ON 
    tracks.TrackId = playlist_track.TrackId
JOIN 
    playlists 
ON 
    playlist_track.PlaylistId = playlists.PlaylistId
WHERE 
    playlists.Name = 'Promotional Hits'
LIMIT 5;
""")
tracks_in_playlist = cursor.fetchall()

# Display the tracks added to the new playlist
for track in tracks_in_playlist:
    print(track[0])

# Commit the transaction
conn.commit()

# Summary

Transactions in databases are a critical concept, ensuring that a series of database operations either all succeed or all fail, maintaining data consistency and integrity. They are particularly useful in scenarios where multiple interdependent operations need to be executed as a single atomic unit. For instance, when updating records across several tables, if one operation fails, the entire transaction can be rolled back to its initial state, as if none of the operations had been performed. This rollback capability is crucial for preventing partial updates that could leave the database in an inconsistent state.

**Summary of key points about transactions:**

* **Atomicity:** Transactions ensure that a series of database operations are treated as a single atomic unit. Either all operations succeed, or none do, preserving the database's consistency.

* **Consistency:** Transactions help maintain database consistency by ensuring that only valid data changes are committed. If a transaction is rolled back, the database reverts to its previous, consistent state.

* **Durability:** Once a transaction is committed, its changes are permanent in the database.

<br>

**Best practices for using transactions**

* **Keep transactions Short:** Long transactions can lock resources for extended periods, leading to bottlenecks. Aim for quick, concise transactions to improve performance and concurrency.

* **Error handling:** Implement robust error handling within transactions to catch failures and roll back changes appropriately, ensuring the database remains in a consistent state.

* **Use transactions judiciously:**  While transactions are powerful, overusing them for operations that don't require atomicity or consistency guarantees can unnecessarily strain the database system.

#  

<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/refs/heads/master/ALX_banners/ALX_Navy.png"  style="width:100px"  ;/>
</div>